# Employee Attrition Prediction — PySpark + Spark MLlib + Gradio

Uses the real IBM HR Analytics Employee Attrition dataset from Kaggle.

In [ ]:
!pip -q install pyspark gradio plotly

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import gradio as gr

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

print("Libraries imported successfully.")

In [ ]:
spark = (
    SparkSession.builder
    .appName("EmployeeAttritionScalableML")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Parallelism:", spark.sparkContext.defaultParallelism)

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_files = [f for f in uploaded if f.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("Upload the Kaggle CSV file.")

csv_path = csv_files[0]
spark_df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

print("Loaded:", csv_path)
print("Rows:", spark_df.count())
print("Columns:", len(spark_df.columns))
spark_df.show(5, truncate=False)

In [ ]:
# Clean target and remove irrelevant columns

spark_df = spark_df.withColumn("Attrition", F.trim(F.col("Attrition")))

print("Original target:")
spark_df.groupBy("Attrition").count().show()

model_df = spark_df.filter(F.col("Attrition").isin(["Yes", "No"]))

drop_cols = ["EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"]
drop_cols = [c for c in drop_cols if c in model_df.columns]
model_df = model_df.drop(*drop_cols)

print("Final target:")
model_df.groupBy("Attrition").count().show()
print("Records:", model_df.count())

In [ ]:
numeric_cols = [
    "Age","DailyRate","DistanceFromHome","Education",
    "EnvironmentSatisfaction","HourlyRate","JobInvolvement","JobLevel",
    "JobSatisfaction","MonthlyIncome","MonthlyRate","NumCompaniesWorked",
    "PercentSalaryHike","PerformanceRating","RelationshipSatisfaction",
    "StockOptionLevel","TotalWorkingYears","TrainingTimesLastYear",
    "WorkLifeBalance","YearsAtCompany","YearsInCurrentRole",
    "YearsSinceLastPromotion","YearsWithCurrManager"
]

categorical_cols = [
    "BusinessTravel","Department","EducationField","Gender",
    "JobRole","MaritalStatus","OverTime"
]

numeric_cols = [c for c in numeric_cols if c in model_df.columns]
categorical_cols = [c for c in categorical_cols if c in model_df.columns]

model_df = model_df.dropna(
    subset=["Attrition"] + numeric_cols + categorical_cols
)

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)
print("Clean records:", model_df.count())

In [ ]:
train_df, test_df = model_df.randomSplit([0.80, 0.20], seed=42)

print("Train:", train_df.count())
print("Test :", test_df.count())

train_df.groupBy("Attrition").count().show()
test_df.groupBy("Attrition").count().show()

In [ ]:
# Spark ML pipeline

indexers = [
    StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_vec")
    for c in categorical_cols
]

label_indexer = StringIndexer(
    inputCol="Attrition",
    outputCol="label",
    handleInvalid="skip"
)

assembler = VectorAssembler(
    inputCols=numeric_cols + [c+"_vec" for c in categorical_cols],
    outputCol="features",
    handleInvalid="keep"
)

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=150,
    maxDepth=6,
    minInstancesPerNode=4,
    featureSubsetStrategy="sqrt",
    seed=42
)

pipeline = Pipeline(
    stages=indexers + encoders + [label_indexer, assembler, rf]
)

model = pipeline.fit(train_df)
print("Model trained successfully.")

In [ ]:
predictions = model.transform(test_df)

predictions.select(
    "Attrition","label","prediction","probability","rawPrediction"
).show(10, truncate=False)

In [ ]:
# Verify binary target and get Yes probability index

label_model = next(
    s for s in model.stages
    if s.__class__.__name__ == "StringIndexerModel"
    and s.getInputCol() == "Attrition"
)

print("Labels:", label_model.labels)

if len(label_model.labels) != 2:
    raise RuntimeError("Target is not binary. Labels found: " + str(label_model.labels))

yes_index = label_model.labels.index("Yes")
print("Yes index:", yes_index)

predictions.select("probability").show(5, truncate=False)

In [ ]:
# Evaluation

def metric(name):
    return MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=name
    ).evaluate(predictions)

accuracy = metric("accuracy")
precision = metric("weightedPrecision")
recall = metric("weightedRecall")
f1 = metric("f1")

roc_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(predictions)

pr_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
).evaluate(predictions)

metrics = pd.DataFrame({
    "Metric": ["Accuracy","Weighted Precision","Weighted Recall","F1","ROC-AUC","PR-AUC"],
    "Score": [accuracy,precision,recall,f1,roc_auc,pr_auc]
})

display(metrics.style.format({"Score":"{:.3f}"}))

In [ ]:
# Confusion matrix

cm = predictions.groupBy("label","prediction").count().orderBy("label","prediction")
cm_pd = cm.toPandas()

cm_pd["Actual"] = cm_pd["label"].map({0.0:"No", 1.0:"Yes"})
cm_pd["Predicted"] = cm_pd["prediction"].map({0.0:"No", 1.0:"Yes"})

display(
    cm_pd.pivot(index="Actual", columns="Predicted", values="count")
    .fillna(0).astype(int)
)

In [ ]:
# Feature importance

rf_model = model.stages[-1]

metadata = predictions.schema["features"].metadata
attrs = metadata["ml_attr"]["attrs"]

all_attrs = []
for key in ["numeric","binary"]:
    all_attrs.extend(attrs.get(key, []))

all_attrs = sorted(all_attrs, key=lambda x: x["idx"])

feature_names = [x["name"] for x in all_attrs]
importance = rf_model.featureImportances.toArray()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values("Importance", ascending=False).head(15)

display(feature_importance)

In [ ]:
fig = px.bar(
    feature_importance.sort_values("Importance"),
    x="Importance",
    y="Feature",
    orientation="h",
    title="Top Features Influencing Employee Attrition"
)
fig.show()

In [ ]:
# Probability check

for row in predictions.select("Attrition","probability").limit(10).collect():
    p = float(row["probability"][yes_index])
    print(row["Attrition"], "-> Yes probability:", f"{p*100:.2f}%")

In [ ]:
# ============================================================
# GRADIO APPLICATION
# ============================================================

# The new employee does NOT have an Attrition column.
# Therefore, use only the fitted feature-processing stages
# and then apply the trained Random Forest model.

feature_models = []

for stage in model.stages:
    is_target_indexer = (
        stage.__class__.__name__ == "StringIndexerModel"
        and stage.getInputCol() == "Attrition"
    )
    is_rf = stage.__class__.__name__ == "RandomForestClassificationModel"

    if not is_target_indexer and not is_rf:
        feature_models.append(stage)


def predict_employee(
    Age, BusinessTravel, Department, DistanceFromHome, Education,
    EducationField, EnvironmentSatisfaction, Gender, JobInvolvement,
    JobLevel, JobRole, JobSatisfaction, MaritalStatus, MonthlyIncome,
    NumCompaniesWorked, OverTime, PercentSalaryHike, PerformanceRating,
    RelationshipSatisfaction, StockOptionLevel, TotalWorkingYears,
    TrainingTimesLastYear, WorkLifeBalance, YearsAtCompany,
    YearsInCurrentRole, YearsSinceLastPromotion, YearsWithCurrManager,
    DailyRate, HourlyRate, MonthlyRate
):
    try:
        employee = {
            "Age": int(Age),
            "BusinessTravel": str(BusinessTravel),
            "Department": str(Department),
            "DistanceFromHome": int(DistanceFromHome),
            "Education": int(Education),
            "EducationField": str(EducationField),
            "EnvironmentSatisfaction": int(EnvironmentSatisfaction),
            "Gender": str(Gender),
            "JobInvolvement": int(JobInvolvement),
            "JobLevel": int(JobLevel),
            "JobRole": str(JobRole),
            "JobSatisfaction": int(JobSatisfaction),
            "MaritalStatus": str(MaritalStatus),
            "MonthlyIncome": int(MonthlyIncome),
            "NumCompaniesWorked": int(NumCompaniesWorked),
            "OverTime": str(OverTime),
            "PercentSalaryHike": int(PercentSalaryHike),
            "PerformanceRating": int(PerformanceRating),
            "RelationshipSatisfaction": int(RelationshipSatisfaction),
            "StockOptionLevel": int(StockOptionLevel),
            "TotalWorkingYears": int(TotalWorkingYears),
            "TrainingTimesLastYear": int(TrainingTimesLastYear),
            "WorkLifeBalance": int(WorkLifeBalance),
            "YearsAtCompany": int(YearsAtCompany),
            "YearsInCurrentRole": int(YearsInCurrentRole),
            "YearsSinceLastPromotion": int(YearsSinceLastPromotion),
            "YearsWithCurrManager": int(YearsWithCurrManager),
            "DailyRate": int(DailyRate),
            "HourlyRate": int(HourlyRate),
            "MonthlyRate": int(MonthlyRate)
        }

        employee_spark = spark.createDataFrame(pd.DataFrame([employee]))

        transformed = employee_spark

        for stage in feature_models:
            transformed = stage.transform(transformed)

        result = (
            rf_model.transform(transformed)
            .select("prediction", "probability")
            .collect()[0]
        )

        prediction_index = int(result["prediction"])
        predicted_class = label_model.labels[prediction_index]

        yes_probability = float(result["probability"][yes_index])
        risk_percentage = yes_probability * 100

        if risk_percentage >= 70:
            risk = "🔴 HIGH RISK"
        elif risk_percentage >= 40:
            risk = "🟠 MEDIUM RISK"
        else:
            risk = "🟢 LOW RISK"

        return f"""# {risk}

## Employee Attrition Probability

### **{risk_percentage:.1f}%**

### Prediction
**{predicted_class}**

---

**Algorithm:** Random Forest Classifier  
**Framework:** Apache Spark MLlib  
**Dataset:** IBM HR Analytics Employee Attrition

> Academic/project demonstration only.
"""

    except Exception as e:
        return f"""# ❌ Prediction Error

`{str(e)}`

Please check the entered values."""


custom_css = """
.gradio-container{max-width:1450px!important;background:#f4f7fb!important;font-family:Inter,Arial,sans-serif}
.hero{background:linear-gradient(135deg,#111827,#312e81,#7c3aed);color:white;border-radius:24px;padding:32px 38px;margin-bottom:20px;box-shadow:0 12px 35px rgba(49,46,129,.22)}
.hero h1{font-size:38px;margin:0 0 8px;font-weight:800}.hero p{margin:0;opacity:.88;font-size:16px}.badges{display:flex;gap:9px;flex-wrap:wrap;margin-top:18px}.badge{background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.2);border-radius:999px;padding:7px 13px;font-size:12px}
.card{background:white;border:1px solid #e5e7eb;border-radius:18px;padding:20px;margin-bottom:14px;box-shadow:0 5px 18px rgba(15,23,42,.05)}
.card h3{margin:0;color:#111827}.desc{color:#6b7280;font-size:13px;margin:4px 0 14px}
.predict-panel{background:linear-gradient(180deg,#fff,#f8fafc);border:1px solid #e2e8f0;border-radius:22px;padding:22px;box-shadow:0 10px 28px rgba(15,23,42,.08)}
.predict-btn{height:56px!important;border-radius:14px!important;font-size:17px!important;font-weight:800!important;background:linear-gradient(90deg,#4f46e5,#7c3aed)!important;border:0!important}
.result{background:white;border:1px solid #e5e7eb;border-radius:18px;padding:22px;text-align:center;box-shadow:0 8px 22px rgba(15,23,42,.06)}
.result .icon{font-size:38px}.result h2{font-size:27px;margin:5px 0;color:#111827}.sub{color:#6b7280;font-size:13px}.prob{background:#f8fafc;border-radius:15px;padding:16px;margin:16px 0}.prob-label{font-size:11px;font-weight:800;letter-spacing:1px;color:#64748b}.prob-value{font-size:46px;font-weight:900;color:#4f46e5}.track{height:10px;background:#e5e7eb;border-radius:99px;overflow:hidden}.fill{height:100%;background:linear-gradient(90deg,#6366f1,#8b5cf6);border-radius:99px}.grid{display:grid;grid-template-columns:1fr 1fr;gap:9px}.item{background:#f8fafc;border-radius:11px;padding:11px;text-align:left}.item span{display:block;color:#94a3b8;font-size:10px}.item b{font-size:13px;color:#1e293b}.note{background:#eef2ff;border:1px solid #c7d2fe;color:#3730a3;border-radius:12px;padding:12px;font-size:12px;margin-top:12px}
"""

with gr.Blocks(title="Employee Attrition Intelligence",theme=gr.themes.Soft(primary_hue="indigo",secondary_hue="violet"),css=custom_css) as demo:
    gr.HTML("""<div class='hero'><h1>👨‍💼 Employee Attrition Intelligence</h1><p>AI-powered attrition prediction using Apache Spark and Random Forest ML.</p><div class='badges'><span class='badge'>⚡ PySpark</span><span class='badge'>🌲 Random Forest</span><span class='badge'>🧠 Spark MLlib</span><span class='badge'>📊 IBM HR Dataset</span><span class='badge'>🚀 Scalable ML</span></div></div>""")
    with gr.Row():
        with gr.Column(scale=7):
            with gr.Group(elem_classes="card"):
                gr.HTML("<h3>👤 Personal Information</h3><div class='desc'>Employee profile and demographic details</div>")
                with gr.Row(): Age=gr.Slider(18,60,value=30,step=1,label="Age"); Gender=gr.Radio(["Male","Female"],value="Male",label="Gender")
                with gr.Row(): MaritalStatus=gr.Dropdown(["Single","Married","Divorced"],value="Single",label="Marital Status"); Education=gr.Slider(1,5,value=3,step=1,label="Education Level")
                with gr.Row(): EducationField=gr.Dropdown(["Life Sciences","Medical","Marketing","Technical Degree","Human Resources","Other"],value="Life Sciences",label="Education Field"); DistanceFromHome=gr.Slider(1,30,value=5,step=1,label="Distance From Home")
            with gr.Group(elem_classes="card"):
                gr.HTML("<h3>🏢 Job & Workplace</h3><div class='desc'>Role, department, travel and workload information</div>")
                with gr.Row(): Department=gr.Dropdown(["Sales","Research & Development","Human Resources"],value="Research & Development",label="Department"); JobRole=gr.Dropdown(["Sales Executive","Research Scientist","Laboratory Technician","Manufacturing Director","Healthcare Representative","Manager","Sales Representative","Research Director","Human Resources"],value="Research Scientist",label="Job Role")
                with gr.Row(): JobLevel=gr.Slider(1,5,value=2,step=1,label="Job Level"); BusinessTravel=gr.Dropdown(["Travel_Rarely","Travel_Frequently","Non-Travel"],value="Travel_Rarely",label="Business Travel")
                with gr.Row(): OverTime=gr.Radio(["Yes","No"],value="No",label="Overtime"); JobInvolvement=gr.Slider(1,4,value=3,step=1,label="Job Involvement"); JobSatisfaction=gr.Slider(1,4,value=3,step=1,label="Job Satisfaction")
            with gr.Group(elem_classes="card"):
                gr.HTML("<h3>💰 Compensation & Performance</h3><div class='desc'>Salary, performance and benefit information</div>")
                with gr.Row(): MonthlyIncome=gr.Number(label="Monthly Income",value=5000,precision=0); DailyRate=gr.Number(label="Daily Rate",value=800,precision=0); HourlyRate=gr.Number(label="Hourly Rate",value=65,precision=0)
                with gr.Row(): MonthlyRate=gr.Number(label="Monthly Rate",value=15000,precision=0); PercentSalaryHike=gr.Slider(11,25,value=15,step=1,label="Salary Hike %"); PerformanceRating=gr.Slider(1,4,value=3,step=1,label="Performance Rating")
                with gr.Row(): StockOptionLevel=gr.Slider(0,3,value=1,step=1,label="Stock Option Level"); EnvironmentSatisfaction=gr.Slider(1,4,value=3,step=1,label="Environment Satisfaction"); RelationshipSatisfaction=gr.Slider(1,4,value=3,step=1,label="Relationship Satisfaction"); WorkLifeBalance=gr.Slider(1,4,value=3,step=1,label="Work-Life Balance")
            with gr.Group(elem_classes="card"):
                gr.HTML("<h3>📊 Career & Experience</h3><div class='desc'>Career history and employee growth indicators</div>")
                with gr.Row(): TotalWorkingYears=gr.Slider(0,40,value=8,step=1,label="Total Working Years"); YearsAtCompany=gr.Slider(0,40,value=5,step=1,label="Years At Company")
                with gr.Row(): YearsInCurrentRole=gr.Slider(0,20,value=3,step=1,label="Years In Current Role"); YearsSinceLastPromotion=gr.Slider(0,15,value=1,step=1,label="Years Since Last Promotion")
                with gr.Row(): YearsWithCurrManager=gr.Slider(0,20,value=2,step=1,label="Years With Current Manager"); NumCompaniesWorked=gr.Slider(0,10,value=2,step=1,label="Companies Worked"); TrainingTimesLastYear=gr.Slider(0,10,value=3,step=1,label="Training Times Last Year")
        with gr.Column(scale=4,elem_classes="predict-panel"):
            gr.HTML("<h2>🔮 Prediction Center</h2><div class='desc'>Analyze the employee profile with the trained Spark ML model.</div>")
            predict_button=gr.Button("🚀  ANALYZE EMPLOYEE",variant="primary",elem_classes="predict-btn")
            output=gr.HTML("<div class='result'><div class='icon'>🤖</div><h2>Ready to Analyze</h2><div class='sub'>Enter details and click Analyze Employee.</div><div class='prob'><div class='prob-label'>ATTRITION PROBABILITY</div><div class='prob-value'>--</div><div class='track'><div class='fill' style='width:0%'></div></div></div></div>")
            gr.HTML("<div class='note'>🔒 <b>Model:</b> Random Forest + Spark MLlib<br>📚 <b>Purpose:</b> Academic / demonstration project</div>")
    predict_button.click(fn=predict_employee,inputs=[Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,DailyRate,HourlyRate,MonthlyRate],outputs=output)
    gr.HTML("<div style='text-align:center;color:#94a3b8;padding:18px;font-size:12px'>Employee Attrition Intelligence • Apache Spark • Spark MLlib • Random Forest</div>")

demo.launch(share=True)

    gr.Markdown("""
# 👨‍💼 Employee Attrition Prediction
### Apache Spark + Random Forest

Enter employee details to estimate attrition probability.
""")

    with gr.Row():
        with gr.Column():
            gr.Markdown("## 👤 Personal Details")
            Age = gr.Number(label="Age", value=30, precision=0)
            Gender = gr.Dropdown(["Male","Female"], value="Male", label="Gender")
            MaritalStatus = gr.Dropdown(["Single","Married","Divorced"], value="Single", label="Marital Status")
            Education = gr.Number(label="Education Level (1-5)", value=3, precision=0)
            EducationField = gr.Dropdown(
                ["Life Sciences","Medical","Marketing","Technical Degree","Human Resources","Other"],
                value="Life Sciences", label="Education Field"
            )
            DistanceFromHome = gr.Number(label="Distance From Home", value=5, precision=0)

        with gr.Column():
            gr.Markdown("## 🏢 Job Details")
            Department = gr.Dropdown(
                ["Sales","Research & Development","Human Resources"],
                value="Research & Development", label="Department"
            )
            JobRole = gr.Dropdown(
                ["Sales Executive","Research Scientist","Laboratory Technician",
                 "Manufacturing Director","Healthcare Representative","Manager",
                 "Sales Representative","Research Director","Human Resources"],
                value="Research Scientist", label="Job Role"
            )
            JobLevel = gr.Number(label="Job Level (1-5)", value=2, precision=0)
            BusinessTravel = gr.Dropdown(
                ["Travel_Rarely","Travel_Frequently","Non-Travel"],
                value="Travel_Rarely", label="Business Travel"
            )
            OverTime = gr.Dropdown(["Yes","No"], value="No", label="Overtime")
            JobInvolvement = gr.Number(label="Job Involvement (1-4)", value=3, precision=0)
            JobSatisfaction = gr.Number(label="Job Satisfaction (1-4)", value=3, precision=0)

    with gr.Row():
        with gr.Column():
            gr.Markdown("## 💰 Salary & Rates")
            MonthlyIncome = gr.Number(label="Monthly Income", value=5000, precision=0)
            DailyRate = gr.Number(label="Daily Rate", value=800, precision=0)
            HourlyRate = gr.Number(label="Hourly Rate", value=65, precision=0)
            MonthlyRate = gr.Number(label="Monthly Rate", value=15000, precision=0)
            PercentSalaryHike = gr.Number(label="Percent Salary Hike", value=15, precision=0)

        with gr.Column():
            gr.Markdown("## 📈 Satisfaction & Performance")
            PerformanceRating = gr.Number(label="Performance Rating (1-4)", value=3, precision=0)
            EnvironmentSatisfaction = gr.Number(label="Environment Satisfaction (1-4)", value=3, precision=0)
            RelationshipSatisfaction = gr.Number(label="Relationship Satisfaction (1-4)", value=3, precision=0)
            WorkLifeBalance = gr.Number(label="Work Life Balance (1-4)", value=3, precision=0)
            StockOptionLevel = gr.Number(label="Stock Option Level (0-3)", value=1, precision=0)
            TrainingTimesLastYear = gr.Number(label="Training Times Last Year", value=3, precision=0)

    with gr.Row():
        with gr.Column():
            gr.Markdown("## 📊 Work Experience")
            TotalWorkingYears = gr.Number(label="Total Working Years", value=8, precision=0)
            YearsAtCompany = gr.Number(label="Years At Company", value=5, precision=0)
            YearsInCurrentRole = gr.Number(label="Years In Current Role", value=3, precision=0)

        with gr.Column():
            gr.Markdown("## 🏆 Career History")
            YearsSinceLastPromotion = gr.Number(label="Years Since Last Promotion", value=1, precision=0)
            YearsWithCurrManager = gr.Number(label="Years With Current Manager", value=2, precision=0)
            NumCompaniesWorked = gr.Number(label="Number of Companies Worked", value=2, precision=0)

    gr.Markdown("---")

    predict_button = gr.Button("🔮 PREDICT EMPLOYEE ATTRITION", variant="primary", size="lg")
    output = gr.Markdown("## 📋 Prediction Result\nEnter details and click Predict.")

    predict_button.click(
        fn=predict_employee,
        inputs=[
            Age, BusinessTravel, Department, DistanceFromHome, Education,
            EducationField, EnvironmentSatisfaction, Gender, JobInvolvement,
            JobLevel, JobRole, JobSatisfaction, MaritalStatus, MonthlyIncome,
            NumCompaniesWorked, OverTime, PercentSalaryHike, PerformanceRating,
            RelationshipSatisfaction, StockOptionLevel, TotalWorkingYears,
            TrainingTimesLastYear, WorkLifeBalance, YearsAtCompany,
            YearsInCurrentRole, YearsSinceLastPromotion, YearsWithCurrManager,
            DailyRate, HourlyRate, MonthlyRate
        ],
        outputs=output
    )

demo.launch(share=True)
